In [2]:
!pip -q install scapy pandas

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
import os

base_dir = "/content/drive/MyDrive/iot/iot device name"

pcap2 = os.path.join(base_dir, "normal_IoT_2.pcap")
pcap3 = os.path.join(base_dir, "normal_IoT_3.pcap")

print(os.path.exists(pcap2), pcap2)
print(os.path.exists(pcap3), pcap3)

True /content/drive/MyDrive/iot/iot device name/normal_IoT_2.pcap
True /content/drive/MyDrive/iot/iot device name/normal_IoT_3.pcap


In [5]:
import pandas as pd
from collections import Counter
from scapy.all import PcapReader, CookedLinux, IP, ARP, TCP, UDP

def is_private_ip(ip):
    if ip is None:
        return False
    ip = str(ip)
    return (
        ip.startswith("192.168.")
        or ip.startswith("10.")
        or any(ip.startswith(f"172.{i}.") for i in range(16, 32))
    )

def bytes_to_mac_addr(x):
    if x is None:
        return None
    if isinstance(x, bytes):
        return ":".join(f"{b:02x}" for b in x[:6])
    if isinstance(x, str):
        return x
    return None

In [6]:
def extract_arp_map(pcap_path):
    counter = Counter()

    with PcapReader(pcap_path) as reader:
        for pkt in reader:
            if ARP in pkt:
                try:
                    mac = pkt[ARP].hwsrc
                    ip = pkt[ARP].psrc
                    if mac and ip and ip != "0.0.0.0":
                        counter[(mac, ip)] += 1
                except:
                    pass

    df = pd.DataFrame([
        {"mac": mac, "ip": ip, "count": cnt}
        for (mac, ip), cnt in counter.items()
    ])

    if not df.empty:
        df = df.sort_values("count", ascending=False).reset_index(drop=True)

    return df

In [7]:
arp_df_2 = extract_arp_map(pcap2)
arp_df_3 = extract_arp_map(pcap3)

print("=== arp_df_2 ===")
display(arp_df_2.head(20))

print("=== arp_df_3 ===")
display(arp_df_3.head(20))

=== arp_df_2 ===


,mac,ip,count
0,a4:91:b1:1e:57:90,192.168.1.1,21744
1,00:c3:f4:0f:67:73,192.168.1.79,11060
2,00:0c:29:d2:b0:02,192.168.1.152,8961
3,00:0c:29:ee:e0:7a,192.168.1.190,4118
4,00:0c:29:a8:3a:da,192.168.1.195,1787
5,60:14:b3:b1:91:73,192.168.1.193,459
6,d4:dc:cd:b4:26:3e,192.168.1.250,67
7,dc:56:e7:5b:61:41,192.168.1.133,20
8,80:3f:5d:10:17:e1,192.168.1.17,17
9,9c:b6:d0:8e:86:79,192.168.1.103,4


=== arp_df_3 ===


,mac,ip,count
0,a4:91:b1:1e:57:90,192.168.1.1,23858
1,00:c3:f4:0f:67:73,192.168.1.79,12555
2,00:0c:29:d2:b0:02,192.168.1.152,11754
3,60:14:b3:b1:91:73,192.168.1.193,4490
4,00:0c:29:ee:e0:7a,192.168.1.190,3903
5,60:14:b3:b1:91:73,192.168.1.192,2152
6,00:0c:29:a8:3a:da,192.168.1.195,1516
7,60:14:b3:b1:91:73,192.168.1.194,1163
8,00:0c:29:7d:af:c3,192.168.1.195,366
9,80:3f:5d:10:17:e1,192.168.1.17,112


In [8]:
def extract_reliable_sll_map(pcap_path):
    counter = Counter()

    with PcapReader(pcap_path) as reader:
        for pkt in reader:
            if CookedLinux in pkt and IP in pkt:
                try:
                    sll = pkt[CookedLinux]
                    lladdrtype = getattr(sll, "lladdrtype", None)
                    lladdrlen = getattr(sll, "lladdrlen", None)
                    sll_mac = bytes_to_mac_addr(getattr(sll, "src", None))
                    ip_src = pkt[IP].src

                    if (
                        lladdrtype == 1 and
                        lladdrlen == 6 and
                        sll_mac is not None and
                        sll_mac != "00:00:00:00:00:00" and
                        is_private_ip(ip_src)
                    ):
                        counter[(sll_mac, ip_src)] += 1
                except:
                    pass

    df = pd.DataFrame([
        {"mac": mac, "ip": ip, "count": cnt}
        for (mac, ip), cnt in counter.items()
    ])

    if not df.empty:
        df = df.sort_values("count", ascending=False).reset_index(drop=True)

    return df

In [9]:
sll_df_2 = extract_reliable_sll_map(pcap2)
sll_df_3 = extract_reliable_sll_map(pcap3)

print("=== sll_df_2 ===")
display(sll_df_2.head(20))

print("=== sll_df_3 ===")
display(sll_df_3.head(20))

=== sll_df_2 ===


,mac,ip,count
0,00:0c:29:d2:b0:02,192.168.1.152,757967
1,00:0c:29:a8:3a:da,192.168.1.195,368712
2,00:0c:29:ee:e0:7a,192.168.1.190,11877
3,00:c3:f4:0f:67:73,192.168.1.79,3818
4,d4:dc:cd:b4:26:3e,192.168.1.250,1098
5,dc:56:e7:5b:61:41,192.168.1.133,875
6,a4:91:b1:1e:57:90,192.168.1.1,696
7,9c:b6:d0:8e:86:79,192.168.1.103,592
8,ec:1f:72:f1:28:6d,192.168.1.6,381
9,60:14:b3:b1:91:73,192.168.1.46,257


=== sll_df_3 ===


,mac,ip,count
0,00:0c:29:d2:b0:02,192.168.1.152,931658
1,00:0c:29:a8:3a:da,192.168.1.195,296249
2,60:14:b3:b1:91:73,192.168.1.192,122931
3,00:0c:29:7d:af:c3,192.168.1.195,72535
4,00:0c:29:ee:e0:7a,192.168.1.190,63550
5,60:14:b3:b1:91:73,192.168.1.193,52611
6,00:c3:f4:0f:67:73,192.168.1.79,4316
7,80:3f:5d:10:17:e1,192.168.1.30,2569
8,dc:56:e7:5b:61:41,192.168.1.133,1149
9,d4:dc:cd:b4:26:3e,192.168.1.250,904


In [10]:
arp2 = arp_df_2.copy()
arp3 = arp_df_3.copy()
sll2 = sll_df_2.copy()
sll3 = sll_df_3.copy()

arp2["source"] = "arp_2"
arp3["source"] = "arp_3"
sll2["source"] = "sll_2"
sll3["source"] = "sll_3"

all_maps = pd.concat([
    arp2[["ip", "mac", "count", "source"]],
    arp3[["ip", "mac", "count", "source"]],
    sll2[["ip", "mac", "count", "source"]],
    sll3[["ip", "mac", "count", "source"]],
], ignore_index=True)

ip_mac_sum = (
    all_maps.groupby(["ip", "mac"], as_index=False)["count"]
    .sum()
    .sort_values(["ip", "count"], ascending=[True, False])
)

num_macs_per_ip = (
    ip_mac_sum.groupby("ip")["mac"]
    .nunique()
    .reset_index(name="num_macs")
)

best_map = (
    ip_mac_sum.drop_duplicates("ip")
    .rename(columns={"mac": "best_mac", "count": "best_count"})
)

total_count = (
    ip_mac_sum.groupby("ip")["count"]
    .sum()
    .reset_index(name="total_count")
)

device_map_df = (
    best_map.merge(num_macs_per_ip, on="ip", how="left")
            .merge(total_count, on="ip", how="left")
)

device_map_df["best_ratio"] = device_map_df["best_count"] / device_map_df["total_count"]

def confidence_rule(row):
    if row["num_macs"] == 1:
        return "high"
    elif row["best_ratio"] >= 0.85:
        return "medium"
    else:
        return "low"

device_map_df["confidence"] = device_map_df.apply(confidence_rule, axis=1)

device_map_df = device_map_df.sort_values(
    ["confidence", "total_count"], ascending=[True, False]
).reset_index(drop=True)

display(device_map_df.head(30))

,ip,best_mac,best_count,num_macs,total_count,best_ratio,confidence
0,192.168.1.152,00:0c:29:d2:b0:02,1710340,1,1710340,1.00000,high
1,192.168.1.192,60:14:b3:b1:91:73,125095,1,125095,1.00000,high
2,192.168.1.190,00:0c:29:ee:e0:7a,83448,1,83448,1.00000,high
3,192.168.1.193,60:14:b3:b1:91:73,57810,1,57810,1.00000,high
4,192.168.1.1,a4:91:b1:1e:57:90,46868,1,46868,1.00000,high
5,192.168.1.79,00:c3:f4:0f:67:73,31749,1,31749,1.00000,high
6,192.168.1.30,80:3f:5d:10:17:e1,2581,1,2581,1.00000,high
7,192.168.1.250,d4:dc:cd:b4:26:3e,2101,1,2101,1.00000,high
8,192.168.1.133,dc:56:e7:5b:61:41,2056,1,2056,1.00000,high
9,192.168.1.194,60:14:b3:b1:91:73,1334,1,1334,1.00000,high


In [11]:
train_device_df = device_map_df[
    (device_map_df["confidence"] == "high") &
    (device_map_df["ip"] != "192.168.1.1")   # 去掉网关
].copy().reset_index(drop=True)

train_device_df["device_name"] = [f"device_{i+1}" for i in range(len(train_device_df))]

display(train_device_df)

,ip,best_mac,best_count,num_macs,total_count,best_ratio,confidence,device_name
0,192.168.1.152,00:0c:29:d2:b0:02,1710340,1,1710340,1.0,high,device_1
1,192.168.1.192,60:14:b3:b1:91:73,125095,1,125095,1.0,high,device_2
2,192.168.1.190,00:0c:29:ee:e0:7a,83448,1,83448,1.0,high,device_3
3,192.168.1.193,60:14:b3:b1:91:73,57810,1,57810,1.0,high,device_4
4,192.168.1.79,00:c3:f4:0f:67:73,31749,1,31749,1.0,high,device_5
5,192.168.1.30,80:3f:5d:10:17:e1,2581,1,2581,1.0,high,device_6
6,192.168.1.250,d4:dc:cd:b4:26:3e,2101,1,2101,1.0,high,device_7
7,192.168.1.133,dc:56:e7:5b:61:41,2056,1,2056,1.0,high,device_8
8,192.168.1.194,60:14:b3:b1:91:73,1334,1,1334,1.0,high,device_9
9,192.168.1.103,9c:b6:d0:8e:86:79,1096,1,1096,1.0,high,device_10


In [12]:
def extract_ip_flows(pcap_path):
    flow_counter = Counter()

    with PcapReader(pcap_path) as reader:
        for pkt in reader:
            if IP not in pkt:
                continue

            src_ip = pkt[IP].src
            dst_ip = pkt[IP].dst

            if TCP in pkt:
                sport = int(pkt[TCP].sport)
                dport = int(pkt[TCP].dport)
                proto = "TCP"
            elif UDP in pkt:
                sport = int(pkt[UDP].sport)
                dport = int(pkt[UDP].dport)
                proto = "UDP"
            else:
                sport = None
                dport = None
                proto = str(pkt[IP].proto)

            flow_counter[(src_ip, dst_ip, sport, dport, proto)] += 1

    flow_df = pd.DataFrame([
        {
            "src_ip": k[0],
            "dst_ip": k[1],
            "src_port": k[2],
            "dst_port": k[3],
            "proto": k[4],
            "packet_count": v
        }
        for k, v in flow_counter.items()
    ])

    if not flow_df.empty:
        flow_df = flow_df.sort_values("packet_count", ascending=False).reset_index(drop=True)

    return flow_df

In [13]:
flow_df_2 = extract_ip_flows(pcap2)
flow_df_3 = extract_ip_flows(pcap3)

display(flow_df_2.head(20))
display(flow_df_3.head(20))

,src_ip,dst_ip,src_port,dst_port,proto,packet_count
0,192.168.1.152,192.168.1.152,1880.0,40688.0,TCP,404490
1,192.168.1.152,192.168.1.152,40688.0,1880.0,TCP,404484
2,192.168.1.152,3.122.49.24,52976.0,1883.0,TCP,337735
3,3.122.49.24,192.168.1.152,1883.0,52976.0,TCP,313943
4,192.168.1.152,192.168.1.152,53972.0,10502.0,TCP,211854
5,192.168.1.152,192.168.1.195,1880.0,54163.0,TCP,211000
6,192.168.1.195,192.168.1.152,54163.0,1880.0,TCP,191507
7,192.168.1.152,192.168.1.195,1880.0,52786.0,TCP,146497
8,192.168.1.195,192.168.1.152,52786.0,1880.0,TCP,133764
9,192.168.1.152,192.168.1.152,10502.0,53972.0,TCP,105928


,src_ip,dst_ip,src_port,dst_port,proto,packet_count
0,192.168.1.152,3.122.49.24,52976.0,1883.0,TCP,267447
1,3.122.49.24,192.168.1.152,1883.0,52976.0,TCP,252181
2,192.168.1.152,192.168.1.152,34296.0,10502.0,TCP,201203
3,192.168.1.152,192.168.1.195,1880.0,49773.0,TCP,190061
4,192.168.1.195,192.168.1.152,49773.0,1880.0,TCP,174500
5,192.168.1.152,192.168.1.152,1880.0,51782.0,TCP,145534
6,192.168.1.152,192.168.1.152,51782.0,1880.0,TCP,145364
7,192.168.1.152,192.168.1.195,1880.0,51323.0,TCP,133628
8,192.168.1.152,192.168.1.152,1880.0,40688.0,TCP,133619
9,192.168.1.152,192.168.1.152,40688.0,1880.0,TCP,133521


In [14]:
ip_to_device = dict(zip(train_device_df["ip"], train_device_df["device_name"]))
ip_to_mac = dict(zip(train_device_df["ip"], train_device_df["best_mac"]))

def label_flow(flow_df):
    df = flow_df.copy()

    def get_local_ip(row):
        if is_private_ip(row["src_ip"]):
            return row["src_ip"]
        elif is_private_ip(row["dst_ip"]):
            return row["dst_ip"]
        return None

    df["local_ip"] = df.apply(get_local_ip, axis=1)
    df["device_name"] = df["local_ip"].map(ip_to_device)
    df["device_mac"] = df["local_ip"].map(ip_to_mac)

    def get_direction(row):
        if pd.isna(row["local_ip"]):
            return "nonlocal"
        elif row["src_ip"] == row["local_ip"]:
            return "out"
        else:
            return "in"

    df["direction"] = df.apply(get_direction, axis=1)
    df["is_labeled"] = df["device_name"].notna()

    return df

In [18]:
labeled_flow_df_2 = label_flow(flow_df_2)
labeled_flow_df_3 = label_flow(flow_df_3)

display(labeled_flow_df_2.head(30))
display(labeled_flow_df_3.head(30))

,src_ip,dst_ip,src_port,dst_port,proto,packet_count,local_ip,device_name,device_mac,direction,is_labeled
0,192.168.1.152,192.168.1.152,1880.0,40688.0,TCP,404490,192.168.1.152,device_1,00:0c:29:d2:b0:02,out,True
1,192.168.1.152,192.168.1.152,40688.0,1880.0,TCP,404484,192.168.1.152,device_1,00:0c:29:d2:b0:02,out,True
2,192.168.1.152,3.122.49.24,52976.0,1883.0,TCP,337735,192.168.1.152,device_1,00:0c:29:d2:b0:02,out,True
3,3.122.49.24,192.168.1.152,1883.0,52976.0,TCP,313943,192.168.1.152,device_1,00:0c:29:d2:b0:02,in,True
4,192.168.1.152,192.168.1.152,53972.0,10502.0,TCP,211854,192.168.1.152,device_1,00:0c:29:d2:b0:02,out,True
5,192.168.1.152,192.168.1.195,1880.0,54163.0,TCP,211000,192.168.1.152,device_1,00:0c:29:d2:b0:02,out,True
6,192.168.1.195,192.168.1.152,54163.0,1880.0,TCP,191507,192.168.1.195,NaN,NaN,out,False
7,192.168.1.152,192.168.1.195,1880.0,52786.0,TCP,146497,192.168.1.152,device_1,00:0c:29:d2:b0:02,out,True
8,192.168.1.195,192.168.1.152,52786.0,1880.0,TCP,133764,192.168.1.195,NaN,NaN,out,False
9,192.168.1.152,192.168.1.152,10502.0,53972.0,TCP,105928,192.168.1.152,device_1,00:0c:29:d2:b0:02,out,True


,src_ip,dst_ip,src_port,dst_port,proto,packet_count,local_ip,device_name,device_mac,direction,is_labeled
0,192.168.1.152,3.122.49.24,52976.0,1883.0,TCP,267447,192.168.1.152,device_1,00:0c:29:d2:b0:02,out,True
1,3.122.49.24,192.168.1.152,1883.0,52976.0,TCP,252181,192.168.1.152,device_1,00:0c:29:d2:b0:02,in,True
2,192.168.1.152,192.168.1.152,34296.0,10502.0,TCP,201203,192.168.1.152,device_1,00:0c:29:d2:b0:02,out,True
3,192.168.1.152,192.168.1.195,1880.0,49773.0,TCP,190061,192.168.1.152,device_1,00:0c:29:d2:b0:02,out,True
4,192.168.1.195,192.168.1.152,49773.0,1880.0,TCP,174500,192.168.1.195,NaN,NaN,out,False
5,192.168.1.152,192.168.1.152,1880.0,51782.0,TCP,145534,192.168.1.152,device_1,00:0c:29:d2:b0:02,out,True
6,192.168.1.152,192.168.1.152,51782.0,1880.0,TCP,145364,192.168.1.152,device_1,00:0c:29:d2:b0:02,out,True
7,192.168.1.152,192.168.1.195,1880.0,51323.0,TCP,133628,192.168.1.152,device_1,00:0c:29:d2:b0:02,out,True
8,192.168.1.152,192.168.1.152,1880.0,40688.0,TCP,133619,192.168.1.152,device_1,00:0c:29:d2:b0:02,out,True
9,192.168.1.152,192.168.1.152,40688.0,1880.0,TCP,133521,192.168.1.152,device_1,00:0c:29:d2:b0:02,out,True


In [19]:
train_flow_df_2 = labeled_flow_df_2[labeled_flow_df_2["is_labeled"]].copy()
train_flow_df_3 = labeled_flow_df_3[labeled_flow_df_3["is_labeled"]].copy()

display(train_flow_df_2.head(20))
display(train_flow_df_3.head(20))

print("train_flow_df_2:", train_flow_df_2.shape)
print("train_flow_df_3:", train_flow_df_3.shape)

,src_ip,dst_ip,src_port,dst_port,proto,packet_count,local_ip,device_name,device_mac,direction,is_labeled
0,192.168.1.152,192.168.1.152,1880.0,40688.0,TCP,404490,192.168.1.152,device_1,00:0c:29:d2:b0:02,out,True
1,192.168.1.152,192.168.1.152,40688.0,1880.0,TCP,404484,192.168.1.152,device_1,00:0c:29:d2:b0:02,out,True
2,192.168.1.152,3.122.49.24,52976.0,1883.0,TCP,337735,192.168.1.152,device_1,00:0c:29:d2:b0:02,out,True
3,3.122.49.24,192.168.1.152,1883.0,52976.0,TCP,313943,192.168.1.152,device_1,00:0c:29:d2:b0:02,in,True
4,192.168.1.152,192.168.1.152,53972.0,10502.0,TCP,211854,192.168.1.152,device_1,00:0c:29:d2:b0:02,out,True
5,192.168.1.152,192.168.1.195,1880.0,54163.0,TCP,211000,192.168.1.152,device_1,00:0c:29:d2:b0:02,out,True
7,192.168.1.152,192.168.1.195,1880.0,52786.0,TCP,146497,192.168.1.152,device_1,00:0c:29:d2:b0:02,out,True
9,192.168.1.152,192.168.1.152,10502.0,53972.0,TCP,105928,192.168.1.152,device_1,00:0c:29:d2:b0:02,out,True
10,192.168.1.152,192.168.1.195,1880.0,51323.0,TCP,47120,192.168.1.152,device_1,00:0c:29:d2:b0:02,out,True
12,192.168.1.250,224.0.0.251,5353.0,5353.0,UDP,1069,192.168.1.250,device_7,d4:dc:cd:b4:26:3e,out,True


,src_ip,dst_ip,src_port,dst_port,proto,packet_count,local_ip,device_name,device_mac,direction,is_labeled
0,192.168.1.152,3.122.49.24,52976.0,1883.0,TCP,267447,192.168.1.152,device_1,00:0c:29:d2:b0:02,out,True
1,3.122.49.24,192.168.1.152,1883.0,52976.0,TCP,252181,192.168.1.152,device_1,00:0c:29:d2:b0:02,in,True
2,192.168.1.152,192.168.1.152,34296.0,10502.0,TCP,201203,192.168.1.152,device_1,00:0c:29:d2:b0:02,out,True
3,192.168.1.152,192.168.1.195,1880.0,49773.0,TCP,190061,192.168.1.152,device_1,00:0c:29:d2:b0:02,out,True
5,192.168.1.152,192.168.1.152,1880.0,51782.0,TCP,145534,192.168.1.152,device_1,00:0c:29:d2:b0:02,out,True
6,192.168.1.152,192.168.1.152,51782.0,1880.0,TCP,145364,192.168.1.152,device_1,00:0c:29:d2:b0:02,out,True
7,192.168.1.152,192.168.1.195,1880.0,51323.0,TCP,133628,192.168.1.152,device_1,00:0c:29:d2:b0:02,out,True
8,192.168.1.152,192.168.1.152,1880.0,40688.0,TCP,133619,192.168.1.152,device_1,00:0c:29:d2:b0:02,out,True
9,192.168.1.152,192.168.1.152,40688.0,1880.0,TCP,133521,192.168.1.152,device_1,00:0c:29:d2:b0:02,out,True
10,192.168.1.192,192.168.1.152,40571.0,1880.0,TCP,121974,192.168.1.192,device_2,60:14:b3:b1:91:73,out,True


train_flow_df_2: (14551, 11)
train_flow_df_3: (15259, 11)


In [ ]:
import pandas as pd

arp2 = arp_df_2.copy()
arp3 = arp_df_3.copy()
sll2 = sll_df_2.copy()
sll3 = sll_df_3.copy()

arp2["source"] = "arp_2"
arp3["source"] = "arp_3"
sll2["source"] = "sll_2"
sll3["source"] = "sll_3"


all_maps = pd.concat([
    arp2[["ip", "mac", "count", "source"]],
    arp3[["ip", "mac", "count", "source"]],
    sll2[["ip", "mac", "count", "source"]],
    sll3[["ip", "mac", "count", "source"]],
], ignore_index=True)

# Maximum number of IP-MAC pairs supported
ip_mac_sum = (
    all_maps.groupby(["ip", "mac"], as_index=False)["count"]
    .sum()
    .sort_values(["ip", "count"], ascending=[True, False])
)

# How many MAC addresses are associated with each IP address?
num_macs_per_ip = (
    ip_mac_sum.groupby("ip")["mac"]
    .nunique()
    .reset_index(name="num_macs")
)

# Primary MAC address for each IP
best_map = (
    ip_mac_sum.drop_duplicates("ip")
    .rename(columns={"mac": "best_mac", "count": "best_count"})
)

# Total number of users per IP address
total_count = (
    ip_mac_sum.groupby("ip")["count"]
    .sum()
    .reset_index(name="total_count")
)

device_map_df = (
    best_map.merge(num_macs_per_ip, on="ip", how="left")
            .merge(total_count, on="ip", how="left")
)

device_map_df["best_ratio"] = device_map_df["best_count"] / device_map_df["total_count"]

def confidence_rule(row):
    if row["num_macs"] == 1:
        return "high"
    elif row["best_ratio"] >= 0.85:
        return "medium"
    else:
        return "low"

device_map_df["confidence"] = device_map_df.apply(confidence_rule, axis=1)

device_map_df = device_map_df.sort_values(
    ["confidence", "total_count"], ascending=[True, False]
).reset_index(drop=True)

display(device_map_df.head(30))

,ip,best_mac,best_count,num_macs,total_count,best_ratio,confidence
0,192.168.1.152,00:0c:29:d2:b0:02,1710340,1,1710340,1.00000,high
1,192.168.1.192,60:14:b3:b1:91:73,125095,1,125095,1.00000,high
2,192.168.1.190,00:0c:29:ee:e0:7a,83448,1,83448,1.00000,high
3,192.168.1.193,60:14:b3:b1:91:73,57810,1,57810,1.00000,high
4,192.168.1.1,a4:91:b1:1e:57:90,46868,1,46868,1.00000,high
5,192.168.1.79,00:c3:f4:0f:67:73,31749,1,31749,1.00000,high
6,192.168.1.30,80:3f:5d:10:17:e1,2581,1,2581,1.00000,high
7,192.168.1.250,d4:dc:cd:b4:26:3e,2101,1,2101,1.00000,high
8,192.168.1.133,dc:56:e7:5b:61:41,2056,1,2056,1.00000,high
9,192.168.1.194,60:14:b3:b1:91:73,1334,1,1334,1.00000,high


In [21]:
mac_device_df = (
    ip_mac_sum.groupby("mac")
    .agg(
        total_count=("count", "sum"),
        num_ips=("ip", "nunique"),
        ip_list=("ip", lambda s: "; ".join(sorted(s.unique())))
    )
    .reset_index()
    .sort_values(["total_count"], ascending=False)
    .reset_index(drop=True)
)

display(mac_device_df.head(30))

,mac,total_count,num_ips,ip_list
0,00:0c:29:d2:b0:02,1710340,1,192.168.1.152
1,00:0c:29:a8:3a:da,668264,1,192.168.1.195
2,60:14:b3:b1:91:73,184708,4,192.168.1.192; 192.168.1.193; 192.168.1.194; 1...
3,00:0c:29:ee:e0:7a,83448,1,192.168.1.190
4,00:0c:29:7d:af:c3,72901,1,192.168.1.195
5,a4:91:b1:1e:57:90,46868,1,192.168.1.1
6,00:c3:f4:0f:67:73,31749,1,192.168.1.79
7,80:3f:5d:10:17:e1,3349,3,192.168.1.17; 192.168.1.191; 192.168.1.30
8,d4:dc:cd:b4:26:3e,2101,1,192.168.1.250
9,dc:56:e7:5b:61:41,2056,1,192.168.1.133


In [22]:
train_ip_map_df = device_map_df[
    (device_map_df["confidence"] == "high") &
    (device_map_df["ip"] != "192.168.1.1")
].copy()

display(train_ip_map_df)

,ip,best_mac,best_count,num_macs,total_count,best_ratio,confidence
0,192.168.1.152,00:0c:29:d2:b0:02,1710340,1,1710340,1.0,high
1,192.168.1.192,60:14:b3:b1:91:73,125095,1,125095,1.0,high
2,192.168.1.190,00:0c:29:ee:e0:7a,83448,1,83448,1.0,high
3,192.168.1.193,60:14:b3:b1:91:73,57810,1,57810,1.0,high
5,192.168.1.79,00:c3:f4:0f:67:73,31749,1,31749,1.0,high
6,192.168.1.30,80:3f:5d:10:17:e1,2581,1,2581,1.0,high
7,192.168.1.250,d4:dc:cd:b4:26:3e,2101,1,2101,1.0,high
8,192.168.1.133,dc:56:e7:5b:61:41,2056,1,2056,1.0,high
9,192.168.1.194,60:14:b3:b1:91:73,1334,1,1334,1.0,high
10,192.168.1.103,9c:b6:d0:8e:86:79,1096,1,1096,1.0,high


In [25]:
final_service_df = pd.DataFrame([
    {"service_name": "service_1", "best_mac": "00:0c:29:d2:b0:02", "ip_list": "192.168.1.152"},
    {"service_name": "service_2", "best_mac": "60:14:b3:b1:91:73", "ip_list": "192.168.1.192; 192.168.1.193; 192.168.1.194; 192.168.1.146"},
    {"service_name": "service_3", "best_mac": "00:0c:29:ee:e0:7a", "ip_list": "192.168.1.190"},
    {"service_name": "service_4", "best_mac": "00:c3:f4:0f:67:73", "ip_list": "192.168.1.79"},
    {"service_name": "service_5", "best_mac": "80:3f:5d:10:17:e1", "ip_list": "192.168.1.17; 192.168.1.191; 192.168.1.30"},
    {"service_name": "service_6", "best_mac": "d4:dc:cd:b4:26:3e", "ip_list": "192.168.1.250"},
    {"service_name": "service_7", "best_mac": "dc:56:e7:5b:61:41", "ip_list": "192.168.1.133"},
])

display(final_service_df)

,service_name,best_mac,ip_list
0,service_1,00:0c:29:d2:b0:02,192.168.1.152
1,service_2,60:14:b3:b1:91:73,192.168.1.192; 192.168.1.193; 192.168.1.194; 1...
2,service_3,00:0c:29:ee:e0:7a,192.168.1.190
3,service_4,00:c3:f4:0f:67:73,192.168.1.79
4,service_5,80:3f:5d:10:17:e1,192.168.1.17; 192.168.1.191; 192.168.1.30
5,service_6,d4:dc:cd:b4:26:3e,192.168.1.250
6,service_7,dc:56:e7:5b:61:41,192.168.1.133


In [26]:
rows = []
for _, r in final_service_df.iterrows():
    for ip in r["ip_list"].split("; "):
        rows.append({
            "ip": ip.strip(),
            "best_mac": r["best_mac"],
            "service_name": r["service_name"]
        })

final_ip_map_df = pd.DataFrame(rows)
display(final_ip_map_df.sort_values("ip").reset_index(drop=True))

,ip,best_mac,service_name
0,192.168.1.133,dc:56:e7:5b:61:41,service_7
1,192.168.1.146,60:14:b3:b1:91:73,service_2
2,192.168.1.152,00:0c:29:d2:b0:02,service_1
3,192.168.1.17,80:3f:5d:10:17:e1,service_5
4,192.168.1.190,00:0c:29:ee:e0:7a,service_3
5,192.168.1.191,80:3f:5d:10:17:e1,service_5
6,192.168.1.192,60:14:b3:b1:91:73,service_2
7,192.168.1.193,60:14:b3:b1:91:73,service_2
8,192.168.1.194,60:14:b3:b1:91:73,service_2
9,192.168.1.250,d4:dc:cd:b4:26:3e,service_6


In [27]:
def is_private_ip(ip):
    if ip is None:
        return False
    ip = str(ip)
    return (
        ip.startswith("192.168.")
        or ip.startswith("10.")
        or any(ip.startswith(f"172.{i}.") for i in range(16, 32))
    )

ip_to_service = dict(zip(final_ip_map_df["ip"], final_ip_map_df["service_name"]))
ip_to_mac = dict(zip(final_ip_map_df["ip"], final_ip_map_df["best_mac"]))

def label_flow_with_final_7(flow_df):
    df = flow_df.copy()

    def get_local_ip(row):
        if is_private_ip(row["src_ip"]):
            return row["src_ip"]
        elif is_private_ip(row["dst_ip"]):
            return row["dst_ip"]
        return None

    df["local_ip"] = df.apply(get_local_ip, axis=1)
    df["service_name"] = df["local_ip"].map(ip_to_service)
    df["service_mac"] = df["local_ip"].map(ip_to_mac)

    def get_direction(row):
        if pd.isna(row["local_ip"]):
            return "nonlocal"
        elif row["src_ip"] == row["local_ip"]:
            return "out"
        else:
            return "in"

    df["direction"] = df.apply(get_direction, axis=1)
    df["is_labeled"] = df["service_name"].notna()

    return df

In [28]:
labeled_flow_df_2 = label_flow_with_final_7(flow_df_2)
labeled_flow_df_3 = label_flow_with_final_7(flow_df_3)

train_flow_df_2 = labeled_flow_df_2[labeled_flow_df_2["is_labeled"]].copy()
train_flow_df_3 = labeled_flow_df_3[labeled_flow_df_3["is_labeled"]].copy()

display(train_flow_df_2.head(20))
display(train_flow_df_3.head(20))

print(train_flow_df_2["service_name"].value_counts())
print(train_flow_df_3["service_name"].value_counts())

,src_ip,dst_ip,src_port,dst_port,proto,packet_count,local_ip,service_name,service_mac,direction,is_labeled
0,192.168.1.152,192.168.1.152,1880.0,40688.0,TCP,404490,192.168.1.152,service_1,00:0c:29:d2:b0:02,out,True
1,192.168.1.152,192.168.1.152,40688.0,1880.0,TCP,404484,192.168.1.152,service_1,00:0c:29:d2:b0:02,out,True
2,192.168.1.152,3.122.49.24,52976.0,1883.0,TCP,337735,192.168.1.152,service_1,00:0c:29:d2:b0:02,out,True
3,3.122.49.24,192.168.1.152,1883.0,52976.0,TCP,313943,192.168.1.152,service_1,00:0c:29:d2:b0:02,in,True
4,192.168.1.152,192.168.1.152,53972.0,10502.0,TCP,211854,192.168.1.152,service_1,00:0c:29:d2:b0:02,out,True
5,192.168.1.152,192.168.1.195,1880.0,54163.0,TCP,211000,192.168.1.152,service_1,00:0c:29:d2:b0:02,out,True
7,192.168.1.152,192.168.1.195,1880.0,52786.0,TCP,146497,192.168.1.152,service_1,00:0c:29:d2:b0:02,out,True
9,192.168.1.152,192.168.1.152,10502.0,53972.0,TCP,105928,192.168.1.152,service_1,00:0c:29:d2:b0:02,out,True
10,192.168.1.152,192.168.1.195,1880.0,51323.0,TCP,47120,192.168.1.152,service_1,00:0c:29:d2:b0:02,out,True
12,192.168.1.250,224.0.0.251,5353.0,5353.0,UDP,1069,192.168.1.250,service_6,d4:dc:cd:b4:26:3e,out,True


,src_ip,dst_ip,src_port,dst_port,proto,packet_count,local_ip,service_name,service_mac,direction,is_labeled
0,192.168.1.152,3.122.49.24,52976.0,1883.0,TCP,267447,192.168.1.152,service_1,00:0c:29:d2:b0:02,out,True
1,3.122.49.24,192.168.1.152,1883.0,52976.0,TCP,252181,192.168.1.152,service_1,00:0c:29:d2:b0:02,in,True
2,192.168.1.152,192.168.1.152,34296.0,10502.0,TCP,201203,192.168.1.152,service_1,00:0c:29:d2:b0:02,out,True
3,192.168.1.152,192.168.1.195,1880.0,49773.0,TCP,190061,192.168.1.152,service_1,00:0c:29:d2:b0:02,out,True
5,192.168.1.152,192.168.1.152,1880.0,51782.0,TCP,145534,192.168.1.152,service_1,00:0c:29:d2:b0:02,out,True
6,192.168.1.152,192.168.1.152,51782.0,1880.0,TCP,145364,192.168.1.152,service_1,00:0c:29:d2:b0:02,out,True
7,192.168.1.152,192.168.1.195,1880.0,51323.0,TCP,133628,192.168.1.152,service_1,00:0c:29:d2:b0:02,out,True
8,192.168.1.152,192.168.1.152,1880.0,40688.0,TCP,133619,192.168.1.152,service_1,00:0c:29:d2:b0:02,out,True
9,192.168.1.152,192.168.1.152,40688.0,1880.0,TCP,133521,192.168.1.152,service_1,00:0c:29:d2:b0:02,out,True
10,192.168.1.192,192.168.1.152,40571.0,1880.0,TCP,121974,192.168.1.192,service_2,60:14:b3:b1:91:73,out,True


service_name
service_1    5761
service_3    5306
service_4    3466
service_2       5
service_5       3
service_7       2
service_6       2
Name: count, dtype: int64
service_name
service_1    6213
service_3    5030
service_4    3940
service_5      45
service_2      21
service_6       2
service_7       2
Name: count, dtype: int64
